# 2 · Distributions
*Data Visualization for Scientists & Public Health Professionals*

A single summary number — a mean, a median — is a compression, and compression loses information. The distribution is what it lost: the *shape* of a variable, where its mass sits, how far its tail reaches, whether it has one peak or two. This notebook covers the four tools for seeing that shape — histogram, KDE, boxplot, violin — and the two judgment calls that quietly decide what story a distribution tells: the **bin width**, and whether to trust a **smooth curve** drawn over few points.

### Learning objectives
- Read a variable's shape from a histogram, and choose a defensible bin width
- Overlay a KDE, and say where it misleads
- Summarize a distribution with a boxplot, and compare distributions across groups
- Show shape and summary together with a violin
- Recognize skew and reach for a log axis when a long tail hides the data

### Agenda
1. The shape behind a summary
2. Bin width is a choice
3. KDE — and where it misleads
4. The boxplot
5. Violins and comparing groups
6. Skew and the log axis

### How the exercises work
Each exercise has a prompt, an empty cell to try it yourself, and a collapsed **Solution** you can expand to check your work.

## Setup

Same `diabetes_viz` data as the last notebook — about 102,000 diabetic hospital encounters, read from the course repository.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

BASE_URL = "https://raw.githubusercontent.com/jimcody2014/2026-python-data/refs/heads/main"
df = pd.read_csv(f"{BASE_URL}/diabetes_viz.csv", low_memory=False)
df.shape

## 1. The shape behind a summary

How long do these patients stay? The mean length of stay is 4.4 days and the median is 4 — two summaries that agree, so it is tempting to stop there. But agreement between mean and median does not mean the job is done. Draw the actual distribution and a different picture appears: most patients leave within a few days, and a long right tail stretches out to two weeks. Neither summary number told you that.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
sns.histplot(df["time_in_hospital"], binwidth=1, color="steelblue", ax=ax)

ax.axvline(df["time_in_hospital"].mean(), color="crimson",
           label=f"mean = {df['time_in_hospital'].mean():.1f}")
ax.axvline(df["time_in_hospital"].median(), color="black", linestyle="--",
           label=f"median = {df['time_in_hospital'].median():.0f}")

ax.set_title("Length of stay: most leave within a few days")
ax.set_xlabel("Days in hospital")
ax.set_ylabel("Encounters")
ax.legend()
fig.tight_layout()
plt.show()

> **Note:** The two summary lines sit almost on top of each other, yet the histogram shows what they cannot: a peak at two to three days and a tail reaching fourteen. The shape carries the clinical story — a typical short stay, with a minority of long, complex ones — that no single number holds. [seaborn histplot](https://seaborn.pydata.org/generated/seaborn.histplot.html)

## 2. Bin width is a choice

A histogram sorts values into bins and counts them, so the **bin width** decides how much detail you see. Too wide, and distinct features smear into one block; too narrow, and real structure drowns in single-count noise. There is no single correct width — it is a judgment call, and the same data can tell noticeably different stories depending on how you set it.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

sns.histplot(df["num_lab_procedures"], binwidth=30, color="steelblue", ax=axes[0])
axes[0].set_title("Too wide (binwidth=30): shape is lost")
axes[0].set_xlabel("Lab procedures per encounter")
axes[0].set_ylabel("Encounters")

sns.histplot(df["num_lab_procedures"], binwidth=5, color="steelblue", ax=axes[1])
axes[1].set_title("binwidth=5: the shape is clear")
axes[1].set_xlabel("Lab procedures per encounter")

fig.tight_layout()
plt.show()

> **Note:** The wide bins collapse everything into a handful of blocks that could hide almost anything; the narrower ones reveal that lab counts cluster in the forties with a gentle spread. The fastest way to build intuition for this trade-off is to stop reasoning about it and *move* it.

### Feel the bin width

Run the cell and drag the slider. Watch the distribution go from spiky noise at small widths to an over-smoothed lump at large ones — and notice that the "right" answer is a *range* of reasonable widths, not a single value. (The slider needs a live kernel; a statically viewed notebook shows only the starting frame.)

In [ ]:
from ipywidgets import interact, IntSlider

def hist_binwidth(binwidth=5):
    fig, ax = plt.subplots(figsize=(9, 4))
    sns.histplot(df["num_lab_procedures"], binwidth=binwidth, color="steelblue", ax=ax)
    ax.set_title(f"Lab procedures — bin width = {binwidth}")
    ax.set_xlabel("Lab procedures per encounter")
    ax.set_ylabel("Encounters")
    fig.tight_layout()
    plt.show()

interact(hist_binwidth, binwidth=IntSlider(min=1, max=40, step=1, value=5));

## 3. KDE — and where it misleads

A **kernel density estimate** draws a smooth curve for the distribution instead of bars. It is excellent for overlaying several distributions on one axis (bars would collide), and it sidesteps the bin-width question. But a KDE is a *model* with its own assumptions, and on a large, clean sample it tracks the histogram closely.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
sns.histplot(df["num_lab_procedures"], binwidth=5, stat="density", color="steelblue", ax=ax)
sns.kdeplot(df["num_lab_procedures"], color="crimson", linewidth=2, ax=ax)
ax.set_title("On large data, the KDE tracks the histogram")
ax.set_xlabel("Lab procedures per encounter")
ax.set_ylabel("Density")
fig.tight_layout()
plt.show()

The danger is small samples. Take one rare specialty — infectious diseases, a few dozen encounters — and the smooth curve starts *inventing* peaks the data cannot support. Put the honest histogram beside it.

In [ ]:
rare = df[df["medical_specialty"] == "InfectiousDiseases"]   # a few dozen encounters

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True)

sns.histplot(rare["num_lab_procedures"], binwidth=10, color="steelblue", ax=axes[0])
axes[0].set_title(f"Histogram (n={len(rare)}): honest and blocky")
axes[0].set_xlabel("Lab procedures per encounter")
axes[0].set_ylabel("Encounters")

sns.kdeplot(rare["num_lab_procedures"], fill=True, color="crimson", ax=axes[1])
axes[1].set_title(f"KDE (n={len(rare)}): smooth, but inventing bumps")
axes[1].set_xlabel("Lab procedures per encounter")

fig.tight_layout()
plt.show()

> **Warning:** Those KDE peaks look authoritative, but they are artifacts of smoothing a handful of points — reshape the sample slightly and they move. Smooth does not mean true. On small samples, trust the blocky histogram over the elegant curve. [seaborn kdeplot](https://seaborn.pydata.org/generated/seaborn.kdeplot.html)

## 4. The boxplot

A **boxplot** throws away shape to show a compact five-number summary: the median line, a box spanning the middle 50% (the interquartile range), whiskers reaching to 1.5 × IQR, and individual points beyond as candidate outliers. It is the most information-dense way to summarize a distribution, and it is built for comparison.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
sns.boxplot(x=df["time_in_hospital"], color="steelblue", ax=ax)
ax.set_title("Length of stay: five-number summary")
ax.set_xlabel("Days in hospital")
fig.tight_layout()
plt.show()

> **Note:** Those 1.5 × IQR whiskers are exactly the outlier rule you used to *clean* data in the pandas course — here you see what it was measuring. The cost of a boxplot is that it hides shape: a box cannot tell a single hump from two. That is what the violin restores.

### Exercise 1 — Shape and summary together *(6 min)*

For **`num_medications`**, draw a histogram (pick a bin width) on top and a boxplot underneath, sharing the x-axis — use `plt.subplots(2, 1, sharex=True)`. In a comment, say whether the variable is symmetric or skewed, and which single number — mean or median — describes it more honestly.

In [ ]:
# Your work here


<details>
<summary><b>Solution</b></summary>

```python
fig, axes = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

sns.histplot(df["num_medications"], binwidth=5, color="steelblue", ax=axes[0])
axes[0].set_title("Number of medications per encounter")
axes[0].set_ylabel("Encounters")

sns.boxplot(x=df["num_medications"], color="steelblue", ax=axes[1])
axes[1].set_xlabel("Number of medications")

fig.tight_layout()
plt.show()
# Right-skewed: the mean (16.0) sits above the median (15), and the upper whisker and
# outliers stretch far to the right. For a skewed variable the median is the more honest summary.
```

**Why this works.** Stacking a histogram over a boxplot on a shared x-axis lets each answer what the other cannot — the histogram shows the shape, the box locates the quartiles and flags the long right tail as outlier points. The rightward stretch is the visual signature of skew, and skew is exactly when the mean gets pulled away from the bulk of the data and the median holds steadier.

</details>

## 5. Violins and comparing groups

A **violin** is a boxplot with a KDE mirrored on each side, so it shows the five-number summary *and* the shape — including a second peak a box would hide. Both the boxplot and the violin come into their own when you put a category on one axis and a number on the other: now you are comparing whole distributions across groups in a single glance. Here, medication counts split by whether the patient was prescribed a diabetes medication.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.violinplot(data=df, x="diabetesMed", y="num_medications", color="steelblue", ax=ax)
ax.set_title("Medication count by whether a diabetes med was prescribed")
ax.set_xlabel("Diabetes medication prescribed")
ax.set_ylabel("Number of medications")
fig.tight_layout()
plt.show()

> **Note:** Each violin shows its group's full shape, so you can see not just that patients on diabetes medication receive more drugs overall, but how the two distributions differ in spread and peak — a comparison a pair of bars could never make. Because the violin is a KDE, the small-sample caveat from Section 3 still applies; with tens of thousands of rows here, it is on solid ground. [seaborn violinplot](https://seaborn.pydata.org/generated/seaborn.violinplot.html)

### Exercise 2 — Do the groups differ? *(8 min)*

Compare **`time_in_hospital`** across the three **`readmitted`** groups (`NO`, `>30`, `<30`) with a boxplot or violin — category on x, days on y. Pass `order=["NO", ">30", "<30"]` so the groups read in a sensible sequence. In a comment, say whether the groups genuinely differ or mostly overlap.

In [ ]:
# Your work here


<details>
<summary><b>Solution</b></summary>

```python
readmit_order = ["NO", ">30", "<30"]

fig, ax = plt.subplots(figsize=(8, 4))
sns.violinplot(data=df, x="readmitted", y="time_in_hospital",
               order=readmit_order, color="steelblue", ax=ax)
ax.set_title("Length of stay by readmission status")
ax.set_xlabel("Readmitted")
ax.set_ylabel("Days in hospital")
fig.tight_layout()
plt.show()
# The three distributions overlap heavily, but patients readmitted within 30 days skew
# slightly longer (median 4 days vs 3 for those never readmitted) -- a real but modest difference.
```

**Why this works.** Putting the category on `x` and the numeric on `y` turns one violin into three comparable ones on a shared scale. The heavy overlap is itself the finding: readmission is not cleanly explained by how long the first stay lasted, even though the readmitted-soon group runs a touch longer. Setting `order=` imposes the meaningful sequence instead of leaving it to chance.

</details>

## 6. Skew and the log axis

Some variables are so right-skewed that a linear axis is useless. Prior inpatient visits is one: about two-thirds of encounters are zero, and the rest trail off to twenty-one. On a linear count axis, that giant zero bar flattens everything else into invisibility. A **log scale** on the y-axis compresses the tall and stretches the short, making the whole range legible at once.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(df["number_inpatient"], binwidth=1, color="steelblue", ax=axes[0])
axes[0].set_title("Linear y: the tail is invisible")
axes[0].set_xlabel("Prior inpatient visits")
axes[0].set_ylabel("Encounters")

sns.histplot(df["number_inpatient"], binwidth=1, color="steelblue", ax=axes[1])
axes[1].set_yscale("log")
axes[1].set_title("Log y: the tail becomes readable")
axes[1].set_xlabel("Prior inpatient visits")
axes[1].set_ylabel("Encounters (log scale)")

fig.tight_layout()
plt.show()

> **Note:** On the left, every bar past zero is a sliver against that first spike; on the right, the log axis reveals a clean geometric decline all the way out to the rare patients with many prior stays. One caution worth stating to any audience: a log axis changes how distances read — equal visual steps are now multiplicative — so label it clearly so no one misreads the scale.

## Wrap-up

You can now see a variable's shape four ways and know what each is for: the **histogram** for raw shape (with a bin width you choose deliberately), the **KDE** for a smooth overlay (trusted only when the sample is large), the **boxplot** for a dense five-number summary and clean group comparison, and the **violin** for shape and summary together. And you can spot a skew severe enough to demand a **log axis**. The theme under all of it: a distribution holds a story a summary statistic compresses away, and a few deliberate choices decide whether that story survives.

**Next:** Comparing across categories — bar and count plots, ordering, grouped bars, and the honest-baseline rule — closing with the first part of the capstone on the rural-closures data.